# Bolton Businesses Without a Website
This notebook finds small businesses in **Bolton, England** that have no website listed on OpenStreetMap.

**Just press the ▶ button on each cell in order, or go to Runtime → Run all.**

In [ ]:
import requests, csv, json, time, io
from collections import Counter
from IPython.display import display
import pandas as pd
print('Libraries ready.')

In [ ]:
# Bolton Metropolitan Borough bounding box (south, west, north, east)
BOLTON_BBOX = (53.490, -2.590, 53.670, -2.310)

OVERPASS_URL = 'https://overpass-api.de/api/interpreter'

BUSINESS_TAGS = ['shop', 'amenity', 'office', 'craft', 'leisure', 'tourism']

NON_BUSINESS_AMENITIES = {
    'parking', 'bicycle_parking', 'bench', 'waste_basket', 'post_box',
    'telephone', 'toilets', 'drinking_water', 'fountain', 'recycling',
    'shelter', 'bus_station', 'ferry_terminal', 'taxi', 'car_sharing',
    'charging_station', 'fuel', 'atm', 'vending_machine', 'school',
    'kindergarten', 'college', 'university', 'library', 'place_of_worship',
    'hospital', 'clinic', 'doctors', 'dentist', 'pharmacy', 'police',
    'fire_station', 'post_office', 'community_centre', 'social_facility',
    'nursing_home', 'prison', 'courthouse', 'townhall', 'embassy',
    'grave_yard', 'crematorium',
}
print('Config set.')

In [ ]:
south, west, north, east = BOLTON_BBOX
bbox_str = f'{south},{west},{north},{east}'

tag_blocks = '\n'.join(
    f'  node["name"]["{tag}"]({bbox_str});\n  way["name"]["{tag}"]({bbox_str});'
    for tag in BUSINESS_TAGS
)
query = f'[out:json][timeout:120];\n(\n{tag_blocks}\n);\nout center;'

print('Querying Overpass API (this may take 20-40 seconds)…')
resp = requests.post(OVERPASS_URL, data={'data': query}, timeout=150)
resp.raise_for_status()
data = resp.json()
print(f'Done. Raw elements returned: {len(data["elements"])}')

In [ ]:
def has_website(tags):
    return any(tags.get(k) for k in ('website','contact:website','url','contact:url'))

def extract(element):
    tags = element.get('tags', {})
    if not tags.get('name') or has_website(tags):
        return None
    category, biz_type = None, None
    for tag in BUSINESS_TAGS:
        if tag in tags:
            category, biz_type = tag, tags[tag]
            break
    if category == 'amenity' and biz_type in NON_BUSINESS_AMENITIES:
        return None
    if element.get('type') == 'node':
        lat, lon = element.get('lat',''), element.get('lon','')
    else:
        c = element.get('center', {})
        lat, lon = c.get('lat',''), c.get('lon','')
    parts = [tags.get('addr:housenumber',''), tags.get('addr:street',''), tags.get('addr:city','') or tags.get('addr:town','')]
    return {
        'name':         tags.get('name',''),
        'business_type': biz_type or '',
        'category':     category or '',
        'address':      ', '.join(p for p in parts if p),
        'postcode':     tags.get('addr:postcode',''),
        'phone':        tags.get('phone','') or tags.get('contact:phone',''),
        'email':        tags.get('email','') or tags.get('contact:email',''),
        'opening_hours': tags.get('opening_hours',''),
        'lat': lat, 'lon': lon,
        'osm_id': element.get('id',''), 'osm_type': element.get('type',''),
    }

raw_businesses = [b for e in data['elements'] if (b := extract(e))]

# Deduplicate
seen, businesses = set(), []
for b in raw_businesses:
    key = (b['name'].lower().strip(), b['address'].lower().strip())
    if key not in seen:
        seen.add(key)
        businesses.append(b)

businesses.sort(key=lambda b: b['name'].lower())
print(f'Businesses WITHOUT a website found: {len(businesses)}')

In [ ]:
df = pd.DataFrame(businesses)

print('\nTop 20 business types:')
print(df['business_type'].value_counts().head(20).to_string())

print(f'\nWith phone:   {df["phone"].astype(bool).sum()}')
print(f'With email:   {df["email"].astype(bool).sum()}')
print(f'With address: {df["address"].astype(bool).sum()}')

print('\nFirst 10 results:')
display(df[['name','business_type','address','postcode','phone']].head(10))

In [ ]:
# ── Lead Scoring ─────────────────────────────────────────────────────────────
# Points: phone +2, email +3, address +1, opening_hours +1
# Category bonus: craft/shop +2 (tradespeople & retailers = best prospects)
#                 amenity/office/tourism/leisure +1

HIGH_VALUE_CATS = {'craft', 'shop'}
MED_VALUE_CATS  = {'amenity', 'office', 'tourism', 'leisure'}

def score_lead(row):
    s = 0
    if row['phone']:         s += 2
    if row['email']:         s += 3
    if row['address']:       s += 1
    if row['opening_hours']: s += 1
    if row['category'] in HIGH_VALUE_CATS:  s += 2
    elif row['category'] in MED_VALUE_CATS: s += 1
    return s

df['score'] = df.apply(score_lead, axis=1)
df_scored = df.sort_values('score', ascending=False)

print(f'Total leads: {len(df_scored)}')
print(f'\nScore distribution:')
print(df_scored['score'].value_counts().sort_index(ascending=False).to_string())

print(f'\nTop 10 prospects:')
display(df_scored[['name','category','business_type','address','phone','email','score']].head(10))

In [ ]:
# ── Pitch Email Generator ────────────────────────────────────────────────────
# Generates a personalised cold email for each lead based on their category.
# The email sells a FREE discovery call, not a website upfront.

SENDER_NAME    = 'Laudem Enterprise Ltd'
SENDER_SIGN    = 'The team at Laudem Enterprise Ltd'

TEMPLATES = {
    'craft': {
        'subject': "Getting {name} found on Google",
        'body': (
            "Hi,\n\n"
            "I came across {name} and noticed you don't have a website yet. "
            "For a local {biz_type} in Bolton, that means customers searching "
            "Google right now for help nearby simply can't find you.\n\n"
            "We're {sender} and we help local tradespeople get a simple, "
            "professional website up quickly and affordably. We'd love to offer "
            "you a free 20-minute call to talk through what would work for your "
            "business — no obligation, no sales pressure.\n\n"
            "Would you be up for a quick chat this week?\n\n"
            "Best wishes,\n{sign}"
        ),
    },
    'shop': {
        'subject': "Helping {name} get found online",
        'body': (
            "Hi,\n\n"
            "I noticed {name} doesn't have a website yet. A lot of shoppers "
            "check online before deciding where to visit — without a site, "
            "you're invisible to them before they even leave home.\n\n"
            "We're {sender} and we help local Bolton businesses get online "
            "quickly and affordably. We'd love to offer you a free 20-minute "
            "call to see what might work for you — completely no-obligation.\n\n"
            "Would you be open to a quick chat this week?\n\n"
            "Best wishes,\n{sign}"
        ),
    },
    'amenity': {
        'subject': "{name} — are new customers finding you online?",
        'body': (
            "Hi,\n\n"
            "I came across {name} and noticed there's no website listed for you. "
            "Most people search online before choosing where to eat, drink or "
            "visit — a simple site with your menu, hours and location can make "
            "a real difference to footfall.\n\n"
            "We're {sender} and we help local Bolton businesses get a smart, "
            "affordable web presence. We'd love to offer you a free 20-minute "
            "call to talk it through — no commitment needed.\n\n"
            "Would you be free for a quick chat this week?\n\n"
            "Best wishes,\n{sign}"
        ),
    },
    'office': {
        'subject': "Is {name} visible online?",
        'body': (
            "Hi,\n\n"
            "I noticed {name} doesn't appear to have a website yet. For a "
            "professional business in Bolton, potential clients often search "
            "online first — if you're not there, they'll find a competitor.\n\n"
            "We're {sender} and we help local businesses build a credible, "
            "affordable online presence. We'd love to offer you a free "
            "20-minute call to explore what might work for you — no obligation.\n\n"
            "Would you be open to a quick call this week?\n\n"
            "Best wishes,\n{sign}"
        ),
    },
    'default': {
        'subject': "Getting {name} online — free chat?",
        'body': (
            "Hi,\n\n"
            "I came across {name} and noticed you don't have a website yet. "
            "In today's market, even a simple site can make a big difference "
            "to how many new customers find you.\n\n"
            "We're {sender} and we help local Bolton businesses get online "
            "quickly and affordably. We'd love to offer you a free 20-minute "
            "call to see if we can help — completely no-obligation.\n\n"
            "Would you be up for a quick chat this week?\n\n"
            "Best wishes,\n{sign}"
        ),
    },
}

def make_email(row):
    cat = row['category'] if row['category'] in TEMPLATES else 'default'
    t   = TEMPLATES[cat]
    vals = dict(
        name   = row['name'],
        biz_type = row['business_type'] or row['category'],
        sender = SENDER_NAME,
        sign   = SENDER_SIGN,
    )
    subject = t['subject'].format(**vals)
    body    = t['body'].format(**vals)
    return subject, body

df_scored[['email_subject','email_body']] = df_scored.apply(
    lambda r: pd.Series(make_email(r)), axis=1
)

print('Email drafts generated.')
print('\nSample — top-scored lead:')
top = df_scored.iloc[0]
print(f"\nTO:      {top['email'] or '(no email — use phone)'}")
print(f"SUBJECT: {top['email_subject']}")
print(f"\n{top['email_body']}")

## Phone Script — Laudem Enterprise Ltd

**Goal: book a free 20-minute discovery call. Don't pitch the website yet.**

---

**Opening (when someone answers)**
> *"Hi, is that [Business Name]? Great — my name is [Your Name] from Laudem Enterprise Ltd. I'm a local web design company based in Bolton. Have I caught you at an okay moment?"*

*(If no → "No problem at all — when would be a better time to call back?")*

---

**The hook (10 seconds)**
> *"The reason I'm calling is I was looking at local [trade / shops / restaurants] in Bolton and I noticed [Business Name] doesn't have a website yet. I just wanted to reach out because we've helped quite a few businesses like yours get online and start picking up customers they were missing."*

---

**Soft close — sell the call, not the website**
> *"I'm not trying to sell you anything today — I'd just love to offer you a free 20-minute chat to show you what's possible and whether it would be worth it for you. No obligation at all."*

---

**Handle the main objections**

| Objection | Response |
|---|---|
| "I don't need a website" | *"Totally understand — a lot of businesses feel that way. The free call just takes 20 minutes and might give you a useful picture of what customers are searching for. If it's not for you, no hard feelings."* |
| "I can't afford it" | *"That's exactly why the call is free — we work with all budgets and there's no pressure. Worth a 20-minute chat?"* |
| "I'm too busy" | *"I completely get it. We can do it whenever suits you — even early morning or after hours."* |
| "I already have someone" | *"Great, sounds like you're sorted then! If anything changes, feel free to give us a call."* |

---

**Booking the call**
> *"Brilliant — I can do [day] at [time] or [day] at [time], whichever suits you better?"*

*(Confirm name, number, email — send a calendar invite immediately after.)*

In [ ]:
output_file = 'bolton_leads_with_pitches.csv'

cols = ['score','name','category','business_type','address','postcode',
        'phone','email','opening_hours','email_subject','email_body','lat','lon']
df_scored[cols].to_csv(output_file, index=False)

print(f'Saved {len(df_scored)} leads to {output_file}')
print(f'\nLeads with score ≥ 5 (best prospects): {(df_scored["score"] >= 5).sum()}')
print(f'Leads with email address:               {df_scored["email"].astype(bool).sum()}')
print(f'Leads with phone number:                {df_scored["phone"].astype(bool).sum()}')
print('\nCSV includes personalised email subject + body for every lead.')

from google.colab import files
files.download(output_file)